In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer

# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(df.head())

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

In [2]:
# Check missing values
print(df.isnull().sum())

# Split features and target
X = df.drop('target', axis=1)
y = df['target']

mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
target                     0
dtype: int64


In [3]:

correlation = df.corr(numeric_only=True)['target'].abs().sort_values(ascending=False)

top_corr_features = correlation[1:6]  # skip target itself
print("Top features (Correlation):")
print(top_corr_features)

Top features (Correlation):
worst concave points    0.793566
worst perimeter         0.782914
mean concave points     0.776614
worst radius            0.776454
mean perimeter          0.742636
Name: target, dtype: float64


In [4]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)
selector.fit(X)

selected_features = X.columns[selector.get_support()]

print("Selected features (Variance Threshold):")
print(selected_features[:5])

Selected features (Variance Threshold):
Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'radius error'],
      dtype='object')


In [5]:
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

# Chi2 requires non-negative values
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

chi2_selector = SelectKBest(score_func=chi2, k=5)
chi2_selector.fit(X_scaled, y)

chi2_features = X.columns[chi2_selector.get_support()]
print("Top features (Chi-Square):")
print(chi2_features)

Top features (Chi-Square):
Index(['mean concavity', 'mean concave points', 'worst perimeter',
       'worst area', 'worst concave points'],
      dtype='object')


In [6]:
from sklearn.feature_selection import f_classif

anova_selector = SelectKBest(score_func=f_classif, k=5)
anova_selector.fit(X, y)

anova_features = X.columns[anova_selector.get_support()]
print("Top features (ANOVA F-test):")
print(anova_features)

Top features (ANOVA F-test):
Index(['mean perimeter', 'mean concave points', 'worst radius',
       'worst perimeter', 'worst concave points'],
      dtype='object')


In [7]:
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X, y)

mi_series = pd.Series(mi, index=X.columns)
top_mi_features = mi_series.sort_values(ascending=False).head(5)

print("Top features (Mutual Information):")
print(top_mi_features)

Top features (Mutual Information):
worst perimeter         0.477911
worst area              0.463679
worst radius            0.450869
mean concave points     0.439621
worst concave points    0.437317
dtype: float64


In [8]:
print("\n=== FINAL SELECTED FEATURES ===")

print("\n1. Correlation:\n", list(top_corr_features.index))
print("\n2. Variance Threshold:\n", list(selected_features[:5]))
print("\n3. Chi-Square:\n", list(chi2_features))
print("\n4. ANOVA:\n", list(anova_features))
print("\n5. Mutual Information:\n", list(top_mi_features.index))


=== FINAL SELECTED FEATURES ===

1. Correlation:
 ['worst concave points', 'worst perimeter', 'mean concave points', 'worst radius', 'mean perimeter']

2. Variance Threshold:
 ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'radius error']

3. Chi-Square:
 ['mean concavity', 'mean concave points', 'worst perimeter', 'worst area', 'worst concave points']

4. ANOVA:
 ['mean perimeter', 'mean concave points', 'worst radius', 'worst perimeter', 'worst concave points']

5. Mutual Information:
 ['worst perimeter', 'worst area', 'worst radius', 'mean concave points', 'worst concave points']
